# 市场数据看板日更流水线

面向公网看板的生产更新入口：每日刷新六个模块，发布可交互的指标序列与 367 项指标目录；上游失败时只使用明确标注的旧缓存，绝不生成占位行情。

In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd

cwd = Path.cwd().resolve()
PROJECT_DIR = next(path for path in (cwd, cwd / 'public_dashboard', cwd.parent) if (path / 'pipeline.py').exists())
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from pipeline import MODULE_KEYS, load_json, resolve_catalog_path, run_update, validate_snapshot

## 1. 路径与更新策略

默认缓存 20 小时，适合每日计划任务；只有显式设置 `DASHBOARD_FORCE_REFRESH=1` 才绕过 TTL。付费源不在本 notebook 中调用。

In [ ]:
DATA_DIR = Path(os.environ.get('DASHBOARD_DATA_DIR', PROJECT_DIR / 'data')).resolve()
CATALOG_PATH = resolve_catalog_path(PROJECT_DIR, DATA_DIR, os.environ.get('DASHBOARD_CATALOG_PATH'))
TTL_SECONDS = int(os.environ.get('DASHBOARD_CACHE_TTL_SECONDS', 20 * 60 * 60))
FORCE_REFRESH = os.environ.get('DASHBOARD_FORCE_REFRESH', '').strip().lower() in {'1', 'true', 'yes', 'on'}
OFFLINE_MODE = os.environ.get('DASHBOARD_OFFLINE', '').strip().lower() in {'1', 'true', 'yes', 'on'}
{'data_dir': str(DATA_DIR), 'catalog': CATALOG_PATH.name, 'ttl_hours': TTL_SECONDS / 3600, 'force': FORCE_REFRESH, 'offline': OFFLINE_MODE}

## 2. 执行日更

刷新过程逐源隔离；一个接口失败不会中断其余模块，也不会用零值或随机数补洞。

In [ ]:
snapshot = run_update(
    project_dir=PROJECT_DIR,
    data_dir=DATA_DIR,
    catalog_path=CATALOG_PATH,
    force=FORCE_REFRESH,
    ttl_seconds=TTL_SECONDS,
)
snapshot['summary']

## 3. 合同与质量闸

写盘前后均校验公共合同、有限数值、六模块 `series` 合同及 367 项目录覆盖。

In [ ]:
validate_snapshot(snapshot)
assert snapshot['summary']['fabricated_observations'] == 0
assert list(snapshot['modules']) == list(MODULE_KEYS)
assert all(isinstance(module.get('series'), list) for module in snapshot['modules'].values())
assert all(item.get('status') in {'live', 'stale', 'unavailable'} for module in snapshot['modules'].values() for item in module['series'])
{'status': snapshot['status'], 'generated_at': snapshot['generated_at'], 'as_of': snapshot['as_of'], 'series': sum(len(module['series']) for module in snapshot['modules'].values()), 'source_errors': len(snapshot['quality']['source_errors'])}

## 4. 六模块状态

`partial` 表示部分免费源失败；`stale` 表示本次失败并保留上次有效值；`failed` 表示没有可发布数据。

In [ ]:
module_rows = [
    {'module': key, 'title': value['title'], 'status': value['status'], 'series': len(value['series']), 'kpis': len(value['kpis']), 'charts': len(value['charts']), 'tables': len(value['tables']), 'alerts': len(value['alerts'])}
    for key, value in snapshot['modules'].items()
]
pd.DataFrame(module_rows)

## 5. 367 项目录覆盖

精选日更指标标记为 `live/stale`；未纳入轻量日更的审定指标保留为 `metadata_only`；精选但无有效观测的指标明确为 `unavailable`。

In [ ]:
catalog_status = pd.DataFrame(snapshot['catalog_status'])
assert len(snapshot['catalog']['rows']) == 367
assert len(catalog_status) == 367
assert set(catalog_status['status']) <= {'live', 'stale', 'unavailable', 'metadata_only'}
assert (catalog_status['status'] == 'live').sum() > 0
catalog_status.groupby(['module', 'status'], dropna=False).size().rename('count').reset_index()

## 6. 落盘验收

前端只读 `dashboard_snapshot.json` 与 `indicator_catalog.json`；历史快照默认保留最近 30 份，原子替换期间不会暴露半文件。

In [ ]:
published_snapshot = load_json(DATA_DIR / 'dashboard_snapshot.json')
published_catalog = load_json(DATA_DIR / 'indicator_catalog.json')
validate_snapshot(published_snapshot)
assert len(published_catalog['rows']) == 367
assert all(isinstance(module.get('series'), list) for module in published_snapshot['modules'].values())
{'snapshot_bytes': (DATA_DIR / 'dashboard_snapshot.json').stat().st_size, 'catalog_bytes': (DATA_DIR / 'indicator_catalog.json').stat().st_size, 'history_files': len(list((DATA_DIR / 'history').glob('snapshot_*.json'))), 'series': sum(len(module['series']) for module in published_snapshot['modules'].values()), 'final_status': published_snapshot['status']}